# Week 5 Lab 2: Advanced Linear Regression

## Foreword
This lab expands on Lab 1 by introducing the **Professional Machine Learning Workflow**.

You will work with **multiple features** (Area, Bedrooms, Age) and learn essential preprocessing steps like:
- Train/Test splits
- Feature normalization with `StandardScaler`
- Model evaluation metrics (RMSE, R-Squared)


## Part 1: loading a Real-ish Dataset

Real houses aren't just defined by 'Area'. They have bedrooms, bathrooms, age, and location. We will use a more complex dataset today.

> **Review**: Remember, `X` (Features) is a Matrix, and `y` (Label) is a Vector.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

# 1. Create a Synthetic Real-Estate Dataset
np.random.seed(42)
samples = 100

# Features: Area (sqm), Bedrooms, Age (years)
area = np.random.randint(50, 300, samples)
bedrooms = np.random.randint(1, 6, samples)
age = np.random.randint(0, 50, samples)

# Target: Price (Rands)
# Formula: Base + 10k*Area + 50k*Bedrooms - 2k*Age + Noise
price = 500000 + (10000 * area) + (50000 * bedrooms) - (2000 * age) + np.random.normal(0, 50000, samples)

df = pd.DataFrame({
    'Area': area,
    'Bedrooms': bedrooms,
    'Age': age,
    'Price': price
})

print("--- Data Preview ---")
print(df.head())


## Part 2: Exploratory Data Analysis (EDA)

Before implementing a model, we must check for correlations. Does 'Age' actually lower the price? Does 'Area' raise it?

We use a **Heatmap** to see these relationships instantly.


In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Feature Correlation Matrix")
plt.show()


---

## Part 3: The Preprocessing Pipeline

This is the most critical step in a professional workflow. We cannot just feed raw data into the model anymore.

### 3.1 Feature Separation
Separate the Dataframe into `X` (Features) and `y` (Target).


In [ ]:
X = df[['Area', 'Bedrooms', 'Age']]
y = df['Price']

print(f"Feature Matrix Shape: {X.shape}")
print(f"Target Vector Shape: {y.shape}")


### 3.2 Train / Test Split
We never test a student on the same questions we taught them in class. Similarly, we never evaluate a model on the data it trained on.
We split data: **80% for Training** (learning patterns) and **20% for Testing** (validating performance).


In [ ]:
# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training samples: {len(X_train)}")
print(f"Testing samples: {len(X_test)}")


### 3.3 Normalization (Scaling)
Our features have different scales:
*   Area: 50 - 300
*   Bedrooms: 1 - 5
*   Age: 0 - 50

A model might think 'Area' is more important just because the number is bigger. We standardise everything using `StandardScaler` (Z-score normalization).


In [ ]:
scaler = StandardScaler()

# Fit on TRAIN data only, then transform both
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Scaled Features (First 3 rows):")
print(X_train_scaled[:3])


---

## Part 4: Multiple Linear Regression

Now we train the model. Because we are using `scikit-learn`, the code looks exactly the same as simple regression, but it handles matrices automatically.


In [ ]:
model = LinearRegression()
model.fit(X_train_scaled, y_train)

print("Model Trained!")
print("Intercept (Base Price):", model.intercept_)
print("Coefficients (Weights):", model.coef_)


**Interpretation**:
Look at the coefficients. Positive coefficients mean the feature adds value (Area, Bedrooms). Negative means it reduces value (Age). Does this match our logic?


---

## Part 5: Evaluation

How good is our model? We test it on the **Test Set** (the 20% of houses it has never seen).

We use two main metrics:
1.  **RMSE (Root Mean Squared Error)**: How much is our prediction wrong on average? (In Rands)
2.  **R-Squared ($R^2$)**: How well does our model explain the variance? (1.0 is perfect, 0.0 is useless).


In [ ]:
# 1. Predict on Test Set
y_pred = model.predict(X_test_scaled)

# 2. Calculate Metrics
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"RMSE Error: R {rmse:,.2f}")
print(f"R-Squared Accuracy: {r2:.4f}")


### 5.1 Visualization: Actual vs Predicted
A perfect model would have all dots on the diagonal line.


In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred, color='blue', alpha=0.6)
plt.plot([y.min(), y.max()], [y.min(), y.max()], 'r--', lw=2) # Diagonal line
plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.title("Actual vs Predicted Prices")
plt.grid(True)
plt.show()


## Summary

You have just built a **Professional ML Pipeline**.
1.  Loaded and Cleaned Data.
2.  Split Data to prevent cheating (overfitting).
3.  Scaled Data to fix feature dominance.
4.  Trained a Multi-Variable Model.
5.  Evaluated using industry metrics.

This is the exact same process used to train massive Deep Learning models, just with more complex math in the middle.
